# Recommender


Ovaj notebook služi kako bi dobili preporučena otvaranja za nekog igrača

---


## 1️. Uvoz biblioteka i podešavanje putanje

Prvo podešavamo putanju kako bi mogli uvseti svoje funkcije. Zatim uvozimo potrebne pakete za parsiranje PGN datoteka i vizualizaciju podataka te iz direktorija src uvozimo dvije funkcije koje će nam pomoći u analizi parse_games i extract_features.

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

In [2]:
from parser import parse_games
from clustering import load_clusters, normalize, fit_clusters
from features import extract_features
import recommender
import pandas as pd
import numpy as np

---

## 2. Parsiranje i učitavanje feature-a 

In [3]:
player_games, opening_count, player_opening_stats, num_of_games = parse_games("../data/raw/lichess_db_standard_rated_2013-01.pgn")

player_names, labels = load_clusters("../data/processed/clusters.csv")

---

## 3. Izgradnja sustava za preporuku otvaranja

In [4]:
opening_recommender = recommender.build_recommendations(player_opening_stats, player_names, labels) 

In [5]:
print(opening_recommender)

{np.int64(0): [{'opening': 'Amar Opening', 'win_rate': 1.0, 'total_games': 7}, {'opening': 'Ware Opening', 'win_rate': 0.9230769230769231, 'total_games': 13}, {'opening': "King's Pawn", 'win_rate': 0.9166666666666666, 'total_games': 12}, {'opening': "King's Gambit, Falkbeer Countergambit, Nimzowitsch-Marshall Countergambit", 'win_rate': 0.8, 'total_games': 5}, {'opening': 'Mexican Defense', 'win_rate': 0.8, 'total_games': 10}, {'opening': 'Latvian Gambit Accepted, Main Line', 'win_rate': 0.8, 'total_games': 5}, {'opening': 'Carr Defense', 'win_rate': 0.7272727272727273, 'total_games': 22}, {'opening': "King's Gambit Accepted, Bishop's Gambit, Bogoljubov Variation", 'win_rate': 0.7, 'total_games': 10}, {'opening': "King's Gambit Declined, Petrov's Defense", 'win_rate': 0.6666666666666666, 'total_games': 6}, {'opening': 'Old Indian', 'win_rate': 0.6666666666666666, 'total_games': 6}, {'opening': "King's Gambit Declined, Mafia Defense", 'win_rate': 0.6666666666666666, 'total_games': 6}, {

---

## 4. Testiranje sustava preporuke

Testiramo na novim igračima i na igračima iz baze

In [6]:
for player in player_names[:5]:
    rec_op = recommender.recommend_for_player(player, player_names, labels, opening_recommender)

Igrač: BFG9k
Klaster: 0
Preporučena otvaranja:
Amar Opening | win_rate: 100.0% | partija: 7
Ware Opening | win_rate: 92.3% | partija: 13
King's Pawn | win_rate: 91.7% | partija: 12
Igrač: mamalak
Klaster: 2
Preporučena otvaranja:
Nimzo-Indian Defense | win_rate: 70.5% | partija: 44
Petrov | win_rate: 70.0% | partija: 10
Kadas Opening | win_rate: 66.3% | partija: 181
Igrač: Desmond_Wilson
Klaster: 2
Preporučena otvaranja:
Nimzo-Indian Defense | win_rate: 70.5% | partija: 44
Petrov | win_rate: 70.0% | partija: 10
Kadas Opening | win_rate: 66.3% | partija: 181
Igrač: savinka59
Klaster: 1
Preporučena otvaranja:
Ware Opening | win_rate: 83.3% | partija: 18
Wade Defense | win_rate: 83.3% | partija: 6
Amazon Attack | win_rate: 83.3% | partija: 6
Igrač: Kozakmamay007
Klaster: 2
Preporučena otvaranja:
Nimzo-Indian Defense | win_rate: 70.5% | partija: 44
Petrov | win_rate: 70.0% | partija: 10
Kadas Opening | win_rate: 66.3% | partija: 181


In [7]:
players_feat_vectors = []
for games in player_games.values():
    feat_vector = extract_features(games, num_of_moves=10)
    players_feat_vectors.append(feat_vector)
players_feat_vectors = np.array(players_feat_vectors)
scaled_matrix, data_mean, data_std = normalize(players_feat_vectors)


In [8]:
model, labels = fit_clusters(scaled_matrix, 3)

cluster_0_mean = players_feat_vectors[labels == 0].mean(axis=0)
cluster_1_mean = players_feat_vectors[labels == 1].mean(axis=0)
cluster_2_mean = players_feat_vectors[labels == 2].mean(axis=0)

test_players = {
    "Agresivan": cluster_0_mean,
    "Pozicijski": cluster_1_mean,
    "Pasivni": cluster_2_mean
}

In [9]:
for naziv, features in test_players.items():
    print(f"\n======={naziv}=======")
    rec = recommender.recommend_new_player(features, model, data_mean, data_std, opening_recommender)



=======Agresivan=======
Procijenjeni klaster: 0
Amar Opening | win_rate: 100.0% | partija: 7
Ware Opening | win_rate: 92.3% | partija: 13
King's Pawn | win_rate: 91.7% | partija: 12

=======Pozicijski=======
Procijenjeni klaster: 1
Ware Opening | win_rate: 83.3% | partija: 18
Wade Defense | win_rate: 83.3% | partija: 6
Amazon Attack | win_rate: 83.3% | partija: 6

=======Pasivni=======
Procijenjeni klaster: 2
Nimzo-Indian Defense | win_rate: 70.5% | partija: 44
Petrov | win_rate: 70.0% | partija: 10
Kadas Opening | win_rate: 66.3% | partija: 181
